<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/04_analisis_resultados/L4S_07_visualizacion_predicciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L4S_07 — Visualización de Predicciones · U-Net ResNet-34

**Objetivo:** Analizar visualmente las predicciones del modelo de segmentación U-Net:  
zonas de acierto, falsos positivos, falsos negativos y mapas de confianza.

**Fuente de datos:** Archivos `fold{N}_vis.npz` guardados por `L4S_04_unet_5fold.ipynb`,  
que contienen las primeras 4 muestras del set de validación de cada fold.

| Archivo | Contenido |
|---------|-----------|
| `fold{N}_vis.npz` | `imgs` (4,14,128,128) · `masks` (4,128,128) · `preds` (4,128,128) |


In [ ]:
# ── Celda 0: Entorno y Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

plt.rcParams.update({'figure.dpi': 120})

DRIVE_PATH = '/content/drive/MyDrive/Landslide4Sense'
ROOT = Path(DRIVE_PATH)
OUT_DIR = ROOT / 'results' / 'comparable_literatura' / 'visualizacion_predicciones'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Umbral de decisión (igual que en entrenamiento)
THR = 0.5

# Canales para componer imagen RGB pseudo-natural (Sentinel-2: B4=Rojo, B3=Verde, B2=Azul)
RGB_IDX = [2, 1, 0]   # índices en el tensor de 14 canales

print(f'✅ Salida: {OUT_DIR}')


## 1. Carga de archivos .npz por fold

In [ ]:
# ── Celda 1: Cargar vis.npz de todos los folds ───────────────────────────────
folds_vis = {}
for k in range(1, 6):
    p = ROOT / f'results/comparable_literatura/unet_5fold/fold{k}_vis.npz'
    if p.exists():
        data = np.load(str(p))
        folds_vis[k] = {
            'imgs':  data['imgs'],   # (4, 14, 128, 128)
            'masks': data['masks'],  # (4, 128, 128)
            'preds': data['preds'],  # (4, 128, 128) — probabilidades
        }
        has_ls = (data['masks'].max(axis=(1,2)) > 0).sum()
        print(f'  Fold {k}: {data["imgs"].shape[0]} muestras | {has_ls} con deslizamiento')
    else:
        print(f'  ⚠️  fold{k}_vis.npz no encontrado: {p}')

if not folds_vis:
    raise FileNotFoundError('No se encontró ningún fold_vis.npz. '
                            'Asegúrate de haber corrido L4S_04_unet_5fold.ipynb.')

print(f'\n✅ {len(folds_vis)} folds con datos visuales cargados')


In [ ]:
# ── Celda 2: Helpers de visualización ────────────────────────────────────────
def to_rgb(img_chw, rgb_idx=RGB_IDX):
    """Convierte (14,128,128) → (128,128,3) RGB normalizado [0,1]."""
    rgb = img_chw[rgb_idx, :, :].transpose(1, 2, 0)  # (128,128,3)
    for c in range(3):
        ch = rgb[:, :, c]
        lo, hi = np.percentile(ch, [2, 98])
        rgb[:, :, c] = np.clip((ch - lo) / (hi - lo + 1e-8), 0, 1)
    return rgb

def error_map(mask, pred_bin):
    """Mapa de colores de error: TP=verde, FP=rojo, FN=naranja, TN=fondo."""
    out = np.zeros((*mask.shape, 3), dtype=np.float32)
    tp = (pred_bin == 1) & (mask == 1)
    fp = (pred_bin == 1) & (mask == 0)
    fn = (pred_bin == 0) & (mask == 1)
    out[tp] = [0.2, 0.8, 0.2]   # verde  — acierto
    out[fp] = [0.9, 0.1, 0.1]   # rojo   — falso positivo
    out[fn] = [1.0, 0.6, 0.0]   # naranja — falso negativo
    return out

OVERLAY_ALPHA = 0.45

def overlay(rgb, mask, pred_bin):
    """RGB + máscara GT (contorno azul) + predicción (contorno rojo)."""
    from matplotlib.contour import QuadContourSet
    fig_tmp, ax_tmp = plt.subplots(1, 1, figsize=(2, 2))
    ax_tmp.imshow(rgb)
    if mask.max() > 0:
        ax_tmp.contour(mask,  levels=[0.5], colors=['#3B82F6'], linewidths=1.5)
    if pred_bin.max() > 0:
        ax_tmp.contour(pred_bin, levels=[0.5], colors=['#EF4444'], linewidths=1.5)
    ax_tmp.axis('off')
    fig_tmp.canvas.draw()
    buf = np.frombuffer(fig_tmp.canvas.tostring_rgb(), dtype=np.uint8)
    buf = buf.reshape(fig_tmp.canvas.get_width_height()[::-1] + (3,))
    plt.close(fig_tmp)
    return buf


## 2. Panel 4 columnas: RGB · GT mask · Predicción · Mapa de error

In [ ]:
# ── Celda 3: Visualización 4-panel para un fold ────────────────────────────
FOLD_VIS = 1   # <── cambia para ver otro fold (1..5)

if FOLD_VIS not in folds_vis:
    print(f'⚠️  Fold {FOLD_VIS} no disponible. Folds con datos: {list(folds_vis.keys())}')
else:
    data  = folds_vis[FOLD_VIS]
    imgs  = data['imgs']    # (4, 14, 128, 128)
    masks = data['masks']   # (4, 128, 128)
    probs = data['preds']   # (4, 128, 128)
    preds = (probs >= THR).astype(np.int32)

    n_samples = len(imgs)
    fig, axes = plt.subplots(n_samples, 5, figsize=(18, 4 * n_samples))
    if n_samples == 1: axes = axes[np.newaxis, :]

    col_titles = ['RGB (B4-B3-B2)', 'GT Mask', 'Predicción (prob.)', 'Mapa de error', 'RGB + contornos']
    for j, t in enumerate(col_titles):
        axes[0, j].set_title(t, fontsize=10, fontweight='bold', pad=6)

    for i in range(n_samples):
        rgb     = to_rgb(imgs[i])
        mask    = masks[i]
        pred_b  = preds[i]
        prob    = probs[i]
        err     = error_map(mask, pred_b)

        tp = int(((pred_b == 1) & (mask == 1)).sum())
        fp = int(((pred_b == 1) & (mask == 0)).sum())
        fn = int(((pred_b == 0) & (mask == 1)).sum())
        has_ls = mask.max() > 0
        tag = f'Fold {FOLD_VIS} · S{i+1} · {"LS" if has_ls else "no LS"}'

        # Col 0: RGB
        axes[i, 0].imshow(rgb); axes[i, 0].axis('off')
        axes[i, 0].set_ylabel(tag, fontsize=8, rotation=0, ha='right', va='center', labelpad=80)

        # Col 1: GT mask
        axes[i, 1].imshow(mask, cmap='Greys', vmin=0, vmax=1); axes[i, 1].axis('off')

        # Col 2: Probabilidad
        im = axes[i, 2].imshow(prob, cmap='RdYlGn', vmin=0, vmax=1)
        axes[i, 2].axis('off')
        plt.colorbar(im, ax=axes[i, 2], fraction=0.046, pad=0.04)

        # Col 3: Error map
        axes[i, 3].imshow(rgb, alpha=0.4)
        axes[i, 3].imshow(err, alpha=0.7)
        axes[i, 3].axis('off')
        axes[i, 3].set_xlabel(f'TP={tp} FP={fp} FN={fn}', fontsize=8)

        # Col 4: RGB + contornos GT (azul) y pred (rojo)
        axes[i, 4].imshow(rgb)
        if mask.max() > 0:
            axes[i, 4].contour(mask,   levels=[0.5], colors=['#3B82F6'], linewidths=1.5)
        if pred_b.max() > 0:
            axes[i, 4].contour(pred_b, levels=[0.5], colors=['#EF4444'], linewidths=1.5)
        axes[i, 4].axis('off')

    # Leyenda error map
    from matplotlib.patches import Patch
    patches = [Patch(color=[0.2,0.8,0.2], label='TP (acierto)'),
               Patch(color=[0.9,0.1,0.1], label='FP (falso positivo)'),
               Patch(color=[1.0,0.6,0.0], label='FN (falso negativo)')]
    fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=9,
               bbox_to_anchor=(0.5, -0.02))

    plt.suptitle(f'U-Net ResNet-34 · Fold {FOLD_VIS} · Predicciones vs GT',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    fname = OUT_DIR / f'predicciones_fold{FOLD_VIS}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show(); print(f'Guardado: {fname}')


## 3. Todos los folds — mapa de error consolidado

In [ ]:
# ── Celda 4: Grid de errores para todos los folds ────────────────────────────
n_folds_avail = len(folds_vis)
n_cols = 4   # muestras por fila
fig, axes = plt.subplots(n_folds_avail, n_cols, figsize=(16, 4 * n_folds_avail))
if n_folds_avail == 1: axes = axes[np.newaxis, :]

for row, (fold_k, data) in enumerate(sorted(folds_vis.items())):
    imgs  = data['imgs']
    masks = data['masks']
    probs = data['preds']
    preds = (probs >= THR).astype(np.int32)

    for col in range(n_cols):
        ax = axes[row, col]
        if col >= len(imgs):
            ax.axis('off'); continue

        rgb    = to_rgb(imgs[col])
        mask   = masks[col]
        pred_b = preds[col]
        err    = error_map(mask, pred_b)

        ax.imshow(rgb, alpha=0.45)
        ax.imshow(err, alpha=0.7)
        ax.axis('off')

        tp = int(((pred_b==1)&(mask==1)).sum())
        fp = int(((pred_b==1)&(mask==0)).sum())
        fn = int(((pred_b==0)&(mask==1)).sum())
        has = '🟢' if (mask.max()>0 and tp>0) else ('🔴' if mask.max()>0 else '⬜')
        ax.set_title(f'F{fold_k}-S{col+1} {has}\nTP={tp} FP={fp} FN={fn}', fontsize=8)

plt.suptitle('Mapa de error por fold y muestra (verde=TP · rojo=FP · naranja=FN)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'error_map_todos_folds.png', dpi=150, bbox_inches='tight')
plt.show(); print('Guardado: error_map_todos_folds.png')


## 4. Análisis cuantitativo de errores por muestra

In [ ]:
# ── Celda 5: Tabla de métricas por muestra ───────────────────────────────────
from sklearn.metrics import f1_score, jaccard_score

print('='*70)
print(f'  {"Fold-Muestra":<14} {"Pixeles LS":>10} {"TP":>6} {"FP":>6} {"FN":>6} {"F1":>7} {"IoU":>7}')
print('='*70)

records = []
for fold_k, data in sorted(folds_vis.items()):
    for i in range(len(data['imgs'])):
        mask   = data['masks'][i].ravel().astype(int)
        pred_b = (data['preds'][i] >= THR).ravel().astype(int)
        n_ls   = mask.sum()
        tp = int(((pred_b==1)&(mask==1)).sum())
        fp = int(((pred_b==1)&(mask==0)).sum())
        fn = int(((pred_b==0)&(mask==1)).sum())
        f1  = f1_score(mask, pred_b, zero_division=0) if n_ls > 0 else float('nan')
        iou = jaccard_score(mask, pred_b, zero_division=0) if n_ls > 0 else float('nan')
        key = f'F{fold_k}-S{i+1}'
        f1_s  = f'{f1:.4f}'  if not (isinstance(f1,float) and f1!=f1) else '  —  '
        iou_s = f'{iou:.4f}' if not (isinstance(iou,float) and iou!=iou) else '  —  '
        print(f'  {key:<14} {n_ls:>10} {tp:>6} {fp:>6} {fn:>6} {f1_s:>7} {iou_s:>7}')
        records.append({'fold': fold_k, 'sample': i+1, 'n_ls': n_ls,
                         'tp': tp, 'fp': fp, 'fn': fn, 'f1': f1, 'iou': iou})
print('='*70)

# Mejor y peor predicción (en parches con deslizamiento)
ls_records = [r for r in records if r['n_ls'] > 0 and not (r['f1']!=r['f1'])]
if ls_records:
    mejor = max(ls_records, key=lambda x: x['f1'])
    peor  = min(ls_records, key=lambda x: x['f1'])
    print(f'\n  Mejor predicción: Fold {mejor["fold"]} Muestra {mejor["sample"]}  F1={mejor["f1"]:.4f}')
    print(f'  Peor predicción:  Fold {peor["fold"]} Muestra {peor["sample"]}   F1={peor["f1"]:.4f}')


## 5. Mapa de confianza — incertidumbre del modelo

In [ ]:
# ── Celda 6: Mapas de probabilidad (confianza) ───────────────────────────────
FOLD_CONF = 1   # <── fold a visualizar

if FOLD_CONF not in folds_vis:
    print(f'Fold {FOLD_CONF} no disponible')
else:
    data  = folds_vis[FOLD_CONF]
    probs = data['preds']
    masks = data['masks']
    imgs  = data['imgs']
    n     = len(probs)

    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1: axes = axes[np.newaxis, :]

    for i in range(n):
        rgb  = to_rgb(imgs[i])
        prob = probs[i]
        mask = masks[i]

        # Incertidumbre = distancia al umbral 0.5
        uncertainty = 1 - 2 * np.abs(prob - 0.5)

        axes[i, 0].imshow(rgb); axes[i, 0].axis('off')
        axes[i, 0].set_title(f'S{i+1} — RGB', fontsize=9)

        im1 = axes[i, 1].imshow(prob, cmap='RdYlGn', vmin=0, vmax=1)
        axes[i, 1].axis('off')
        axes[i, 1].set_title(f'S{i+1} — Probabilidad (confianza)', fontsize=9)
        plt.colorbar(im1, ax=axes[i, 1], fraction=0.046)

        im2 = axes[i, 2].imshow(uncertainty, cmap='hot', vmin=0, vmax=1)
        if mask.max() > 0:
            axes[i, 2].contour(mask, levels=[0.5], colors=['#3B82F6'], linewidths=1.5)
        axes[i, 2].axis('off')
        axes[i, 2].set_title(f'S{i+1} — Incertidumbre (blanco=incierto)', fontsize=9)
        plt.colorbar(im2, ax=axes[i, 2], fraction=0.046)

    plt.suptitle(f'U-Net · Fold {FOLD_CONF} — Mapas de confianza e incertidumbre\n'
                 f'(contorno azul = GT mask)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'confianza_fold{FOLD_CONF}.png', dpi=150, bbox_inches='tight')
    plt.show(); print(f'Guardado: confianza_fold{FOLD_CONF}.png')


## 6. Resumen

In [ ]:
# ── Celda 7: Resumen ──────────────────────────────────────────────────────────
import numpy as np
from sklearn.metrics import f1_score

print('='*60)
print('  RESUMEN — VISUALIZACIÓN DE PREDICCIONES U-Net')
print('='*60)
total_tp = total_fp = total_fn = 0
f1_vals  = []
for fold_k, data in sorted(folds_vis.items()):
    for i in range(len(data['imgs'])):
        mask   = data['masks'][i].ravel().astype(int)
        pred_b = (data['preds'][i] >= THR).ravel().astype(int)
        total_tp += int(((pred_b==1)&(mask==1)).sum())
        total_fp += int(((pred_b==1)&(mask==0)).sum())
        total_fn += int(((pred_b==0)&(mask==1)).sum())
        if mask.sum() > 0:
            f1_vals.append(f1_score(mask, pred_b, zero_division=0))

print(f'\n  Muestras analizadas  : {sum(len(d["imgs"]) for d in folds_vis.values())}')
print(f'  Folds disponibles    : {sorted(folds_vis.keys())}')
print(f'  TP total (píxeles)   : {total_tp:,}')
print(f'  FP total (píxeles)   : {total_fp:,}')
print(f'  FN total (píxeles)   : {total_fn:,}')
if f1_vals:
    print(f'  F1 promedio (con LS) : {np.mean(f1_vals):.4f} ± {np.std(f1_vals):.4f}')
print(f'\n  Archivos guardados en: {OUT_DIR}')
print('    - predicciones_fold{N}.png')
print('    - error_map_todos_folds.png')
print('    - confianza_fold{N}.png')
print('='*60)
